### Import Dependencies

In [2]:
import uuid

import httpx
from a2a.client import A2ACardResolver, create_client
from a2a.types import (
    AgentCard,
    Message,
    Part,
    Role,
    SendMessageRequest
)

from google.protobuf.json_format import MessageToJson

### Test Connection to A2A Server

In [4]:
BASE_URL = 'http://localhost:10001'
PUBLIC_AGENT_CARD_PATH = '/.well-known/agent-card.json'

In [5]:
async with httpx.AsyncClient() as httpx_client:

    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=BASE_URL
    )

    try:
        print(f"Fetching public agent card from: {BASE_URL}{PUBLIC_AGENT_CARD_PATH}")
        public_card = await resolver.get_agent_card()
        print("Fetched public agent card")
        print(MessageToJson(public_card, indent=2))

    except Exception as e:
        print(f"Error fetching public agent card: {e}")

Fetching public agent card from: http://localhost:10001/.well-known/agent-card.json
Fetched public agent card
{
  "name": "warehouse_manager_agent",
  "description": "The warehouse manager agent is responsible for checking the availability of items in the warehouse and reserving them.",
  "supportedInterfaces": [
    {
      "url": "http://localhost:10001/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "skills": [
    {
      "id": "ABC",
      "name": "Check Availability",
      "description": "Check availability of items across warehouses.",
      "tags": [
        "warehouse",
        "availability"
      ],
      "examples": [
        "What is the availability of the item 123 ?"
      ]
    },
    {
      "id": "DEF",
      "name": "Reserve Items",
      "description": "Reserve items in the warehou

### Run commands against the Agent

In [6]:
timeout_config = httpx.Timeout(
    connect=10.0,   # Connection timeout
    read=100.0,     # Read timeout (important for long-running operations)
    write=10.0,     # Write timeout
    pool=10.0       # Pool timeout
)

client = await create_client(
    agent=BASE_URL,
    resolver_http_kwargs={"timeout": timeout_config}
)

print("A2AClient initialized")

message_id = str(uuid.uuid4())
message_payload = Message(
    role=Role.ROLE_USER,
    message_id=message_id,
    parts=[Part(text="Hello, how are you?")],
)
request = SendMessageRequest(message=message_payload)

async for response in client.send_message(request):
    print(response)

A2AClient initialized
task {
  id: "82b8b578-3b1d-494c-b9f6-64c50ac88046"
  context_id: "833f9eed-1029-402a-bda4-2b032b11381a"
  status {
    state: TASK_STATE_SUBMITTED
  }
}

status_update {
  task_id: "82b8b578-3b1d-494c-b9f6-64c50ac88046"
  context_id: "833f9eed-1029-402a-bda4-2b032b11381a"
  status {
    state: TASK_STATE_WORKING
    timestamp {
      seconds: 1786973228
      nanos: 180784000
    }
  }
}

artifact_update {
  task_id: "82b8b578-3b1d-494c-b9f6-64c50ac88046"
  context_id: "833f9eed-1029-402a-bda4-2b032b11381a"
  artifact {
    artifact_id: "acd91e7d-a87a-46b4-830e-f847b3df1f70"
    parts {
      text: "Hello! I’m doing well, thanks. How can I help you today?"
    }
  }
}

status_update {
  task_id: "82b8b578-3b1d-494c-b9f6-64c50ac88046"
  context_id: "833f9eed-1029-402a-bda4-2b032b11381a"
  status {
    state: TASK_STATE_COMPLETED
    timestamp {
      seconds: 1786973232
      nanos: 491460000
    }
  }
}



In [7]:
async def run_a2a_warehouse_agent(query: str):

    timeout_config = httpx.Timeout(
        connect=10.0,   # Connection timeout
        read=100.0,     # Read timeout (important for long-running operations)
        write=10.0,     # Write timeout
        pool=10.0       # Pool timeout
    )

    client = await create_client(
        agent=BASE_URL,
        resolver_http_kwargs={"timeout": timeout_config}
    )


    message_id = str(uuid.uuid4())
    message_payload = Message(
        role=Role.ROLE_USER,
        message_id=message_id,
        parts=[Part(text=query)],
    )
    request = SendMessageRequest(message=message_payload)

    final_text_parts = []

    async for response in client.send_message(request):
        if response.HasField("artifact_update"):
            for part in response.artifact_update.artifact.parts:
                if part.text:
                    final_text_parts.append(part.text)

    final_response = "".join(final_text_parts)

    return final_response

In [8]:
answer_1 = await run_a2a_warehouse_agent("What is the availability of B09X1LDMH6 in all of your warehouses?")

In [9]:
print(answer_1)

Availability check completed for **B09X1LDMH6**:

### Warehouses with available stock
- **FR-PAR-01 — Paris Central Depot**: 65 units
- **DE-HAM-01 — Hamburg North Warehouse**: 60 units
- **FR-LYO-01 — Lyon Regional Warehouse**: 28 units
- **DE-MUN-01 — Munich Logistics Hub**: 28 units
- **FR-MAR-01 — Marseille Mediterranean Hub**: 2 units

### Warehouse with no stock
- **DE-BER-01 — Berlin Distribution Center**: 0 units

### Summary
- The item is **fully available** across multiple warehouses.
- No partial fulfillment is needed for a quantity of 1.


In [10]:
answer_1 = await run_a2a_warehouse_agent("Can you reserve 1 of B09X1LDMH6 in Hamburg?")

In [11]:
print(answer_1)

Reservation completed.

Actions performed:
- Checked warehouse availability for 1x B09X1LDMH6
- Reserved 1x B09X1LDMH6 from Hamburg North Warehouse (DE-HAM-01)

Status: Reserved successfully in Hamburg.
